### Oitavas Sulamericana


In [1]:
import cartolafc
import pandas as pd
from difflib import get_close_matches
import json
from pathlib import Path

pd.set_option('display.max_columns', 50)            # permite a visualização de 50 colunas do dataframe
pd.options.display.float_format = '{:.2f}'.format   # pandas: para todos os números aparecerem com duas casas decimais

# Cria uma instância da API
api = cartolafc.Api(attempts=5)

2026-04-27 13:30:05,341 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


### Dicionário com os IDs e Nomes dos times.

In [2]:
def carregar_payload_js(caminho_arquivo, nome_constante):
    conteudo = Path(caminho_arquivo).read_text(encoding="utf-8")
    conteudo_json = conteudo.replace(f"const {nome_constante} = ", "").strip().rstrip(";")
    return json.loads(conteudo_json)


def resolver_caminho_libertadores(nome_arquivo):
    candidatos = [
        Path("libertadores") / "datasets_liberta" / nome_arquivo,
        Path("..") / ".." / "libertadores" / "datasets_liberta" / nome_arquivo,
    ]
    for caminho in candidatos:
        if caminho.exists():
            return caminho
    raise FileNotFoundError(f"Arquivo não encontrado: {nome_arquivo}")


arquivos_origem = [
    (
        resolver_caminho_libertadores("times_3o_lugar_fase_1.js"),
        "timesTerceiroLugarFase1",
    ),
    (
        resolver_caminho_libertadores("times_3o_e_4o_lugar_fase_2.js"),
        "timesTerceiroEQuartoLugarFase2",
    ),
]

participantes_liberta_sula = []
for caminho, constante in arquivos_origem:
    participantes_liberta_sula.extend(carregar_payload_js(caminho, constante))

df_participantes_liberta_sula = pd.DataFrame(participantes_liberta_sula).rename(
    columns={"id": "ID do Time", "nome": "Nome do Time", "origem": "Origem", "grupo": "Grupo", "posicao": "Posição"}
)

if "Posição" not in df_participantes_liberta_sula.columns:
    df_participantes_liberta_sula["Posição"] = pd.NA

df_participantes_liberta_sula.loc[
    df_participantes_liberta_sula["Origem"] == "3o lugar fase 1", "Posição"
] = 3

df_participantes_liberta_sula["Posição"] = pd.to_numeric(
    df_participantes_liberta_sula["Posição"], errors="coerce"
).astype("Int64")

df_participantes_liberta_sula = df_participantes_liberta_sula.sort_values(
    ["Origem", "Grupo", "Posição", "Nome do Time"], na_position="last"
).reset_index(drop=True)

nomes_por_id = {
    int(row["ID do Time"]): row["Nome do Time"]
    for _, row in df_participantes_liberta_sula.dropna(subset=["ID do Time", "Nome do Time"]).iterrows()
}

if not nomes_por_id:
    print("🕒 Ainda não há participantes carregados da Libertadores para a Sul-Americana.")
    COMPETICAO_INICIOU = False
else:
    COMPETICAO_INICIOU = True
    display(df_participantes_liberta_sula[[col for col in ["Origem", "Grupo", "Posição", "ID do Time", "Nome do Time"] if col in df_participantes_liberta_sula.columns]])



,Origem,Grupo,Posição,ID do Time,Nome do Time
0,3o lugar fase 1,Grupo A,3,13951133,JUV. KP
1,3o lugar fase 1,Grupo B,3,30267301,Máquina Laranjja
2,3o lugar fase 1,Grupo C,3,7017989,dasdoresfc
3,3o lugar fase 1,Grupo D,3,24856400,Grêmio imortal 36
4,3o lugar fase 1,Grupo E,3,18344271,FÚRIA LEON
5,3o lugar fase 1,Grupo F,3,18346776,AZURRA82
6,3o lugar fase 1,Grupo G,3,24468241,Grêmio imortal 37
7,3o lugar fase 1,Grupo H,3,13707047,Super Vasco f.c
8,3o lugar fase 2,Grupo I,3,19033717,Mau Humor F.C.
9,3o lugar fase 2,Grupo J,3,117598,A Lenda Super Vasco F.c


In [3]:
if not COMPETICAO_INICIOU:
    print("🕒 Aguardando definição dos participantes. Sem dados para montar confrontos.")
else:
    # Referência do chaveamento manual já validado:
    # Jogo 1: Grupo A fase 1 x 4º Grupo L fase 2
    # Jogo 2: Grupo B fase 1 x 4º Grupo K fase 2
    # Jogo 3: Grupo C fase 1 x 4º Grupo J fase 2
    # Jogo 4: Grupo D fase 1 x 4º Grupo I fase 2
    # Jogo 5: Grupo E fase 1 x 3º Grupo L fase 2
    # Jogo 6: Grupo F fase 1 x 3º Grupo K fase 2
    # Jogo 7: Grupo G fase 1 x 3º Grupo J fase 2
    # Jogo 8: Grupo H fase 1 x 3º Grupo I fase 2

    fase_1_por_grupo = {
        row["Grupo"]: int(row["ID do Time"])
        for _, row in df_participantes_liberta_sula[
            df_participantes_liberta_sula["Origem"] == "3o lugar fase 1"
        ].iterrows()
    }

    fase_2_por_chave = {
        (row["Grupo"], int(row["Posição"])): int(row["ID do Time"])
        for _, row in df_participantes_liberta_sula[
            df_participantes_liberta_sula["Origem"].isin(["3o lugar fase 2", "4o lugar fase 2"])
        ].iterrows()
    }

    pareamentos = [
        ("Jogo 1 (JG1)", "Grupo A", "Grupo L", 4),
        ("Jogo 2 (JG2)", "Grupo B", "Grupo K", 4),
        ("Jogo 3 (JG3)", "Grupo C", "Grupo J", 4),
        ("Jogo 4 (JG4)", "Grupo D", "Grupo I", 4),
        ("Jogo 5 (JG5)", "Grupo E", "Grupo L", 3),
        ("Jogo 6 (JG6)", "Grupo F", "Grupo K", 3),
        ("Jogo 7 (JG7)", "Grupo G", "Grupo J", 3),
        ("Jogo 8 (JG8)", "Grupo H", "Grupo I", 3),
    ]

    dados_torneio = []
    for jogo, grupo_fase_1, grupo_fase_2, posicao_fase_2 in pareamentos:
        id_fase_1 = fase_1_por_grupo.get(grupo_fase_1)
        id_fase_2 = fase_2_por_chave.get((grupo_fase_2, posicao_fase_2))
        if id_fase_1 is None or id_fase_2 is None:
            raise ValueError(
                f"Pareamento incompleto para {jogo}: {grupo_fase_1} x {grupo_fase_2} ({posicao_fase_2}º)"
            )
        dados_torneio.extend([
            (jogo, id_fase_1),
            (jogo, id_fase_2),
        ])
    
    # Criar DataFrame base
    df_torneio = pd.DataFrame(dados_torneio, columns=["Jogo", "ID do Time"])
    
    # Adicionar Nome do Time usando o dicionário
    df_torneio["Nome do Time"] = df_torneio["ID do Time"].map(nomes_por_id)
    
    # Adicionar ID do Jogo
    df_torneio["ID do Jogo"] = df_torneio.groupby("Jogo").cumcount() + 1
    df_torneio["ID do Jogo"] = df_torneio["ID do Jogo"].astype(str) + "_" + df_torneio["Jogo"].str[-2]
    
    
    # Reorganizar colunas
    df_torneio = df_torneio[["Jogo", "ID do Time", "Nome do Time", "ID do Jogo"]]
    
    df_sula_jogo_1 = df_torneio[df_torneio["Jogo"] == "Jogo 1 (JG1)"]
    df_sula_jogo_2 = df_torneio[df_torneio["Jogo"] == "Jogo 2 (JG2)"]
    df_sula_jogo_3 = df_torneio[df_torneio["Jogo"] == "Jogo 3 (JG3)"]
    df_sula_jogo_4 = df_torneio[df_torneio["Jogo"] == "Jogo 4 (JG4)"]
    df_sula_jogo_5 = df_torneio[df_torneio["Jogo"] == "Jogo 5 (JG5)"]
    df_sula_jogo_6 = df_torneio[df_torneio["Jogo"] == "Jogo 6 (JG6)"]
    df_sula_jogo_7 = df_torneio[df_torneio["Jogo"] == "Jogo 7 (JG7)"]
    df_sula_jogo_8 = df_torneio[df_torneio["Jogo"] == "Jogo 8 (JG8)"]
    
    # Lista de jogos
    jogos = {
        "Jogo 1 (JG1)": df_sula_jogo_1,
        "Jogo 2 (JG2)": df_sula_jogo_2,
        "Jogo 3 (JG3)": df_sula_jogo_3,
        "Jogo 4 (JG4)": df_sula_jogo_4,
        "Jogo 5 (JG5)": df_sula_jogo_5,
        "Jogo 6 (JG6)": df_sula_jogo_6,
        "Jogo 7 (JG7)": df_sula_jogo_7,
        "Jogo 8 (JG8)": df_sula_jogo_8
    }
    
    display(df_sula_jogo_2)



,Jogo,ID do Time,Nome do Time,ID do Jogo
2,Jogo 2 (JG2),30267301,Máquina Laranjja,1_2
3,Jogo 2 (JG2),387186,DM Studio,2_2


### Jogos das Oitavas de Final da Sulamericana

In [4]:
# Rodada 13 - Fase 1 Sulamericana (Equivalente a 13º Rodada do Campeonato Brasileiro) 
confrontos_13a_rodada = [
    # Jogo 1 (JG1)
    ("Jogo 1 (JG1)", "1_1", "2_1"),

    # Jogo 2 (JG2)
    ("Jogo 2 (JG2)", "1_2", "2_2"),

    # Jogo 3 (JG3)
    ("Jogo 3 (JG3)", "1_3", "2_3"),

    # Jogo 4 (JG4)
    ("Jogo 4 (JG4)", "1_4", "2_4"),

    # Jogo 5 (JG5)
    ("Jogo 5 (JG5)", "1_5", "2_5"),

    # Jogo 6 (JG6)
    ("Jogo 6 (JG6)", "1_6", "2_6"),

    # Jogo 7 (JG7)
    ("Jogo 7 (JG7)", "1_7", "2_7"),

    # Jogo 8 (JG8)
    ("Jogo 8 (JG8)", "1_8", "2_8")    
]

# Rodada 14 - Fase 1 Sulamericana (Equivalente a 14º Rodada do Campeonato Brasileiro) 
confrontos_14a_rodada = [
    # Jogo 1 (JG1)
    ("Jogo 1 (JG1)", "2_1", "1_1"),

    # Jogo 2 (JG2)
    ("Jogo 2 (JG2)", "2_2", "1_2"),

    # Jogo 3 (JG3)
    ("Jogo 3 (JG3)", "2_3", "1_3"),

    # Jogo 4 (JG4)
    ("Jogo 4 (JG4)", "2_4", "1_4"),

    # Jogo 5 (JG5)
    ("Jogo 5 (JG5)", "2_5", "1_5"),

    # Jogo 6 (JG6)
    ("Jogo 6 (JG6)", "2_6", "1_6"),

    # Jogo 7 (JG7)
    ("Jogo 7 (JG7)", "2_7", "1_7"),

    # Jogo 8 (JG8)
    ("Jogo 8 (JG8)", "2_8", "1_8")    
]

In [5]:
# Transformar em DataFrame
df_confrontos = pd.DataFrame(confrontos_13a_rodada, columns=["Jogo", "Mandante_ID", "Visitante_ID"])

# Junta com df_torneio para buscar dados dos mandantes
df_mandantes = df_torneio.rename(columns={
    "ID do Jogo": "Mandante_ID",
    "Nome do Time": "Mandante_Nome",
    "ID do Time": "Mandante_ID_Time"
})[["Jogo", "Mandante_ID", "Mandante_Nome", "Mandante_ID_Time"]]

# Junta com df_torneio para buscar dados dos visitantes
df_visitantes = df_torneio.rename(columns={
    "ID do Jogo": "Visitante_ID",
    "Nome do Time": "Visitante_Nome",
    "ID do Time": "Visitante_ID_Time"    
})[["Jogo", "Visitante_ID", "Visitante_Nome", "Visitante_ID_Time"]]

display(df_confrontos)

,Jogo,Mandante_ID,Visitante_ID
0,Jogo 1 (JG1),1_1,2_1
1,Jogo 2 (JG2),1_2,2_2
2,Jogo 3 (JG3),1_3,2_3
3,Jogo 4 (JG4),1_4,2_4
4,Jogo 5 (JG5),1_5,2_5
5,Jogo 6 (JG6),1_6,2_6
6,Jogo 7 (JG7),1_7,2_7
7,Jogo 8 (JG8),1_8,2_8


In [6]:
# Transformar em DataFrame
df_confrontos = pd.DataFrame(confrontos_13a_rodada, columns=["Jogo", "Mandante_ID", "Visitante_ID"])
df_confrontos["Rodada"] = 13  
df_rodada_13 = df_confrontos.merge(df_mandantes, on=["Jogo", "Mandante_ID"])
df_rodada_13 = df_rodada_13.merge(df_visitantes, on=["Jogo", "Visitante_ID"])


# Transformar em DataFrame
df_confrontos_14 = pd.DataFrame(confrontos_14a_rodada, columns=["Jogo", "Mandante_ID", "Visitante_ID"])
df_confrontos_14["Rodada"] = 14
df_rodada_14 = df_confrontos_14.merge(df_mandantes, on=["Jogo", "Mandante_ID"])
df_rodada_14 = df_rodada_14.merge(df_visitantes, on=["Jogo", "Visitante_ID"])

display(df_rodada_13)

,Jogo,Mandante_ID,Visitante_ID,Rodada,Mandante_Nome,Mandante_ID_Time,Visitante_Nome,Visitante_ID_Time
0,Jogo 1 (JG1),1_1,2_1,13,JUV. KP,13951133,KillerColorado,36359
1,Jogo 2 (JG2),1_2,2_2,13,Máquina Laranjja,30267301,DM Studio,387186
2,Jogo 3 (JG3),1_3,2_3,13,dasdoresfc,7017989,Tatols Beants F.C,212042
3,Jogo 4 (JG4),1_4,2_4,13,Grêmio imortal 36,24856400,JV5 Tricolor Gaúcho,1747619
4,Jogo 5 (JG5),1_5,2_5,13,FÚRIA LEON,18344271,Texas Club 2026,1273719
5,Jogo 6 (JG6),1_6,2_6,13,AZURRA82,18346776,TORRESMO COM PINGA PRO26.1,47544767
6,Jogo 7 (JG7),1_7,2_7,13,Grêmio imortal 37,24468241,A Lenda Super Vasco F.c,117598
7,Jogo 8 (JG8),1_8,2_8,13,Super Vasco f.c,13707047,Mau Humor F.C.,19033717


In [7]:
df_rodadas = pd.concat([
    df_rodada_13,
    df_rodada_14,
], ignore_index=True)

# Ajustar a numeração da rodada para refletir as rodadas do Cartola (Rodada 13 até 14)
df_rodadas["Rodada"] = df_rodadas["Rodada"]  # Ex: 1 -> 13, 2 -> 14

df_rodadas.to_excel("1_confrontos_oitavas_sula.xlsx", index=False)

# Exibir os confrontos da fase 1
display(df_rodadas.head(20)) 

,Jogo,Mandante_ID,Visitante_ID,Rodada,Mandante_Nome,Mandante_ID_Time,Visitante_Nome,Visitante_ID_Time
0,Jogo 1 (JG1),1_1,2_1,13,JUV. KP,13951133,KillerColorado,36359
1,Jogo 2 (JG2),1_2,2_2,13,Máquina Laranjja,30267301,DM Studio,387186
2,Jogo 3 (JG3),1_3,2_3,13,dasdoresfc,7017989,Tatols Beants F.C,212042
3,Jogo 4 (JG4),1_4,2_4,13,Grêmio imortal 36,24856400,JV5 Tricolor Gaúcho,1747619
4,Jogo 5 (JG5),1_5,2_5,13,FÚRIA LEON,18344271,Texas Club 2026,1273719
5,Jogo 6 (JG6),1_6,2_6,13,AZURRA82,18346776,TORRESMO COM PINGA PRO26.1,47544767
6,Jogo 7 (JG7),1_7,2_7,13,Grêmio imortal 37,24468241,A Lenda Super Vasco F.c,117598
7,Jogo 8 (JG8),1_8,2_8,13,Super Vasco f.c,13707047,Mau Humor F.C.,19033717
8,Jogo 1 (JG1),2_1,1_1,14,KillerColorado,36359,JUV. KP,13951133
9,Jogo 2 (JG2),2_2,1_2,14,DM Studio,387186,Máquina Laranjja,30267301


In [8]:
# Criar lista de dicionários no formato desejado
confrontos_js_oitavas_sula = []

for _, row in df_rodadas.iterrows():
    confronto = {
        "jogo": row["Jogo"],
        "rodada": int(row["Rodada"]),
        "mandante": {
            "id": int(row["Mandante_ID_Time"]),
            "nome": row["Mandante_Nome"]
        },
        "visitante": {
            "id": int(row["Visitante_ID_Time"]),
            "nome": row["Visitante_Nome"]
        }
    }
    confrontos_js_oitavas_sula.append(confronto)

# Converter para JSON formatado
json_str = json.dumps(confrontos_js_oitavas_sula, indent=2, ensure_ascii=False)

# Salvar como arquivo JS com uma variável global
with open("1_confrontos_oitavas_sula.js", "w", encoding="utf-8") as f:
    f.write("const confrontos_oitavas_sula = ")
    f.write(json_str)
    f.write(";")

In [9]:
def exibir_confrontos(df_rodadas, rodada=None, jogo=None):
    """
    Filtra e exibe os confrontos por rodada e/ou grupo.

    Parâmetros:
    - df_rodadas: DataFrame com todos os confrontos
    - rodada: número da rodada (int ou None para todas)
    - grupo: nome do grupo (str ou None para todos)
    
    Retorna:
    - DataFrame filtrado com as colunas relevantes
    """    
    colunas = ["Rodada", "Jogo", "Mandante_ID_Time", "Mandante_Nome", "Visitante_ID_Time", "Visitante_Nome"]
    df_filtrado = df_rodadas.copy()

    df_filtrado["Rodada"] = df_filtrado["Rodada"].astype(str) + "ª Rodada"    
    if rodada is not None:
        df_filtrado = df_filtrado[df_filtrado["Rodada"] == rodada]

    if jogo is not None:
        df_filtrado = df_filtrado[df_filtrado["Jogo"] == jogo]

    return df_filtrado[colunas].sort_values(by=["Jogo", "Rodada"])

In [10]:
jogo = 8

# Exibir todos os confrontos do Grupo A
display(exibir_confrontos(df_rodadas, jogo= f"Jogo {jogo} (JG{jogo})").head())

,Rodada,Jogo,Mandante_ID_Time,Mandante_Nome,Visitante_ID_Time,Visitante_Nome
7,13ª Rodada,Jogo 8 (JG8),13707047,Super Vasco f.c,19033717,Mau Humor F.C.
15,14ª Rodada,Jogo 8 (JG8),19033717,Mau Humor F.C.,13707047,Super Vasco f.c


In [11]:
import requests
import time
import sys

FASE_INICIO = 13
FASE_LIMITE = 14
PER_REQ_SLEEP = 1.0

ROOT = Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.parciais as parciais
parciais.HEADERS = globals().get("HEADERS", {})
fetch_pontuados = parciais.fetch_pontuados
fetch_time_payload = parciais.fetch_time_payload
clubes_que_ja_jogaram = parciais.clubes_que_ja_jogaram
calcular_parcial_time = parciais.calcular_parcial_time

sess = requests.Session()
sess.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})


def http_status_e_rodada():
    r = sess.get("https://api.cartola.globo.com/mercado/status", timeout=20)
    r.raise_for_status()
    data = r.json()
    return int(data.get("status_mercado", 0)), int(data.get("rodada_atual", 0))


def _coerce_float(value):
    try:
        if value is None:
            return None
        return float(value)
    except Exception:
        return None


def extrair_pontuacao_payload(payload):
    if not isinstance(payload, dict):
        return None

    candidatos = []
    for chave in ("pontos", "pontuacao"):
        candidatos.append(payload.get(chave))

    time_info = payload.get("time", {})
    if isinstance(time_info, dict):
        for chave in ("pontos", "pontuacao"):
            candidatos.append(time_info.get(chave))

    pontos_info = payload.get("pontos")
    if isinstance(pontos_info, dict):
        for chave in ("rodada", "total", "valor"):
            candidatos.append(pontos_info.get(chave))

    for candidato in candidatos:
        valor = _coerce_float(candidato)
        if valor is not None:
            return valor

    return None


def obter_pontuacao_rodada_fechada(api, time_id, rodada):
    payload = fetch_time_payload(int(time_id), int(rodada))
    pontuacao_payload = extrair_pontuacao_payload(payload)
    if pontuacao_payload is not None:
        return pontuacao_payload

    time_rodada = api.time(time_id=int(time_id), rodada=int(rodada))
    for atributo in ("pontos", "pontuacao", "ultima_pontuacao"):
        valor = _coerce_float(getattr(time_rodada, atributo, None))
        if valor is not None:
            return valor

    return None


def gerar_df_pontuacoes(api, ids_times):
    global status_http, rodada_http, rodada_api, rod_ref, FASE_FIM, RODADAS_CONCLUIDAS_FIM

    colunas_fase = [f"Rodada {i}" for i in range(FASE_INICIO, FASE_LIMITE + 1)]

    try:
        status_http, rodada_http = http_status_e_rodada()
    except Exception:
        status_http, rodada_http = 0, 0

    try:
        rodada_api = int(api.mercado().rodada_atual)
    except Exception:
        rodada_api = int(rodada_http or 0)

    rod_ref = int(rodada_http or rodada_api or 0)

    if status_http == 2:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, rod_ref))
        if rod_ref > FASE_LIMITE:
            RODADAS_CONCLUIDAS_FIM = FASE_LIMITE
        else:
            RODADAS_CONCLUIDAS_FIM = FASE_FIM - 1
    elif status_http == 1:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, max(rod_ref - 1, FASE_INICIO)))
        RODADAS_CONCLUIDAS_FIM = FASE_FIM
    else:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, rod_ref if rod_ref else FASE_INICIO))
        RODADAS_CONCLUIDAS_FIM = FASE_FIM

    print(
        f"Status={status_http} | rodada_http={rodada_http} | rodada_api={rodada_api} | "
        f"FASE_INICIO={FASE_INICIO} | FASE_FIM={FASE_FIM} | fechadas ate {RODADAS_CONCLUIDAS_FIM}"
    )

    dados = {}
    for nome, time_id in ids_times.items():
        pontuacoes = {}
        if RODADAS_CONCLUIDAS_FIM >= FASE_INICIO:
            for rodada in range(FASE_INICIO, RODADAS_CONCLUIDAS_FIM + 1):
                try:
                    pontuacoes[rodada] = obter_pontuacao_rodada_fechada(api, time_id, rodada)
                except Exception as e:
                    print(f"Erro ao acessar pontuacao da rodada {rodada} para o time {time_id}: {e}")
                    pontuacoes[rodada] = None
        dados[nome] = {f"Rodada {rodada}": pontuacoes.get(rodada) for rodada in range(FASE_INICIO, FASE_LIMITE + 1)}

    df = pd.DataFrame.from_dict(dados, orient="index")
    df = df.reindex(columns=colunas_fase)
    df = df.apply(pd.to_numeric, errors="coerce")

    col_atual = f"Rodada {rod_ref}"
    if status_http == 2 and FASE_INICIO <= rod_ref <= FASE_LIMITE:
        print(f"Rodada {rod_ref} em andamento: aplicando parciais")
        mapa_pontuados = fetch_pontuados()
        if mapa_pontuados:
            clubes_jogaram = clubes_que_ja_jogaram(rod_ref)
            for nome_time, time_id in ids_times.items():
                try:
                    total = calcular_parcial_time(int(time_id), rod_ref, mapa_pontuados, clubes_jogaram)
                    if col_atual not in df.columns:
                        df[col_atual] = pd.NA
                    df.loc[nome_time, col_atual] = round(total, 2)
                except Exception as e:
                    print(f"Erro ao calcular parcial da rodada {rod_ref} para o time {time_id}: {e}")
                time.sleep(PER_REQ_SLEEP)
        else:
            print("Parciais indisponiveis no momento.")
    else:
        print("Sem parciais para aplicar nesta fase.")

    colunas_com_dados = [col for col in df.columns if df[col].notna().any()]
    if colunas_com_dados:
        lideres = {col: df[col].dropna().idxmax() for col in colunas_com_dados}
        df = pd.concat([df, pd.DataFrame([lideres], index=["Lider_Rodada"])], axis=0)

    return df


In [12]:
if not COMPETICAO_INICIOU:
    print("Competição ainda não começou. Pulando geração de pontuações.")
    df_pontuacoes = pd.DataFrame()
else:
    ids_times = {v: k for k, v in nomes_por_id.items()}
    
    df_pontuacoes = gerar_df_pontuacoes(api, ids_times)
    display(df_pontuacoes.T)



Status=1 | rodada_http=14 | rodada_api=14 | FASE_INICIO=13 | FASE_FIM=13 | fechadas ate 13


Sem parciais para aplicar nesta fase.


,JUV. KP,Máquina Laranjja,dasdoresfc,Grêmio imortal 36,FÚRIA LEON,AZURRA82,Grêmio imortal 37,Super Vasco f.c,Mau Humor F.C.,A Lenda Super Vasco F.c,TORRESMO COM PINGA PRO26.1,Texas Club 2026,JV5 Tricolor Gaúcho,Tatols Beants F.C,DM Studio,KillerColorado,Lider_Rodada
Rodada 13,117.04,123.04,119.54,103.07,116.79,99.64,103.59,104.89,122.49,120.79,104.29,106.39,115.44,110.35,104.29,97.64,Máquina Laranjja
Rodada 14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
def classificacao_por_grupo(df_rodadas, df_pontuacoes):
    """Gera classificacao por grupo (ou jogo), exibindo times mesmo com pontuações zeradas."""
    df_pontuacoes_times = df_pontuacoes.drop(index='Lider_Rodada', errors='ignore')
    estatisticas = {}

    for _, confronto in df_rodadas.iterrows():
        rodada = confronto.get("Rodada")
        mandante = confronto.get("Mandante_Nome")
        visitante = confronto.get("Visitante_Nome")
        jogo = confronto.get("Jogo")
        coluna_rodada = f"Rodada {rodada}"

        # Pula se o jogo não existir na planilha
        if mandante not in df_pontuacoes_times.index or visitante not in df_pontuacoes_times.index:
            continue
        if coluna_rodada not in df_pontuacoes_times.columns:
            continue

        pontos_mandante = df_pontuacoes_times.at[mandante, coluna_rodada]
        pontos_visitante = df_pontuacoes_times.at[visitante, coluna_rodada]

        # Se não tiver pontuação ainda, define como 0
        if pd.isnull(pontos_mandante):
            pontos_mandante = 0
        if pd.isnull(pontos_visitante):
            pontos_visitante = 0

        # Inicializa estrutura
        if jogo not in estatisticas:
            estatisticas[jogo] = {}
        for time in [mandante, visitante]:
            if time not in estatisticas[jogo]:
                estatisticas[jogo][time] = {
                    "Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                    "Total_Cartola": 0, "Cartola_Sofrido": 0
                }

        # Atualiza totais
        estatisticas[jogo][mandante]["Total_Cartola"] += pontos_mandante
        estatisticas[jogo][mandante]["Cartola_Sofrido"] += pontos_visitante
        estatisticas[jogo][visitante]["Total_Cartola"] += pontos_visitante
        estatisticas[jogo][visitante]["Cartola_Sofrido"] += pontos_mandante

        # Atualiza pontos somente se jÃƒÂ¡ houve pontuaÃƒÂ§ÃƒÂ£o real
        if pontos_mandante == 0 and pontos_visitante == 0:
            continue

        if pontos_mandante > pontos_visitante:
            estatisticas[jogo][mandante]["Pontos"] += 3
            estatisticas[jogo][mandante]["Vitórias"] += 1
            estatisticas[jogo][visitante]["Derrotas"] += 1
        elif pontos_mandante < pontos_visitante:
            estatisticas[jogo][visitante]["Pontos"] += 3
            estatisticas[jogo][visitante]["Vitórias"] += 1
            estatisticas[jogo][mandante]["Derrotas"] += 1
        else:
            estatisticas[jogo][mandante]["Pontos"] += 1
            estatisticas[jogo][visitante]["Pontos"] += 1
            estatisticas[jogo][mandante]["Empates"] += 1
            estatisticas[jogo][visitante]["Empates"] += 1

    # Caso ainda não existam pontuações, monta estrutura zerada
    if not estatisticas:
        print("Exibindo classificação inicial com pontuações zeradas.")
        jogos = df_rodadas["Jogo"].unique()
        estatisticas = {
            jogo: {
                row["Mandante_Nome"]: {"Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                                       "Total_Cartola": 0, "Cartola_Sofrido": 0},
                row["Visitante_Nome"]: {"Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                                        "Total_Cartola": 0, "Cartola_Sofrido": 0}
            }
            for jogo, row in df_rodadas.groupby("Jogo").first().iterrows()
        }

    # Monta DataFrame final
    df_resultado = pd.concat([
        pd.DataFrame({
            "Jogo": jogo,
            "Nome do Time": list(times.keys()),
            "Pontos": [stats["Pontos"] for stats in times.values()],
            "Vitórias": [stats["Vitórias"] for stats in times.values()],
            "Empates": [stats["Empates"] for stats in times.values()],
            "Derrotas": [stats["Derrotas"] for stats in times.values()],
            "Total Cartola": [stats["Total_Cartola"] for stats in times.values()],
            "Cartola Sofrido": [stats["Cartola_Sofrido"] for stats in times.values()],
            "Saldo Cartola": [
                stats["Total_Cartola"] - stats["Cartola_Sofrido"] for stats in times.values()
            ]
        })
        for jogo, times in estatisticas.items()
    ], ignore_index=True)

    # Ordena e adiciona posição
    df_resultado = df_resultado.sort_values(
        by=["Jogo", "Pontos", "Vitórias", "Total Cartola", "Saldo Cartola"],
        ascending=[True, False, False, False, False]
    )
    df_resultado["Posição"] = df_resultado.groupby("Jogo").cumcount() + 1

    df_resultado_por_jogo = {
        jogo: df_resultado[df_resultado["Jogo"] == jogo] for jogo in df_resultado["Jogo"].unique()
    }

    return df_resultado, df_resultado_por_jogo


In [14]:
# Padroniza os nomes no df_rodadas
df_rodadas["Mandante_Nome"] = df_rodadas["Mandante_Nome"].str.strip()
df_rodadas["Visitante_Nome"] = df_rodadas["Visitante_Nome"].str.strip()

# Padroniza os ÃƒÆ’Ã‚Â­ndices do df_pontuacoes
df_pontuacoes.index = df_pontuacoes.index.str.strip()

# Exibe para conferÃƒÆ’Ã‚?ncia
display(df_pontuacoes)

,Rodada 13,Rodada 14
JUV. KP,117.04,NaN
Máquina Laranjja,123.04,NaN
dasdoresfc,119.54,NaN
Grêmio imortal 36,103.07,NaN
FÚRIA LEON,116.79,NaN
AZURRA82,99.64,NaN
Grêmio imortal 37,103.59,NaN
Super Vasco f.c,104.89,NaN
Mau Humor F.C.,122.49,NaN
A Lenda Super Vasco F.c,120.79,NaN


In [15]:
# Gerar a classificação da fase 1 sula
df_resultado, df_resultado_por_jogo = classificacao_por_grupo(df_rodadas, df_pontuacoes)

# Salvar cada grupo em uma aba do Excel
with pd.ExcelWriter("1_classificacao_por_jogo_oitavas_sula.xlsx") as writer:
    for jogo, df in df_resultado_por_jogo.items():
        df.to_excel(writer, sheet_name=jogo, index=False)

# Exibir a classificação geral
df_resultado_jogo_1 = df_resultado[df_resultado["Jogo"] == "Jogo 1 (JG1)"]
df_resultado_jogo_2 = df_resultado[df_resultado["Jogo"] == "Jogo 2 (JG2)"]
df_resultado_jogo_3 = df_resultado[df_resultado["Jogo"] == "Jogo 3 (JG3)"]
df_resultado_jogo_4 = df_resultado[df_resultado["Jogo"] == "Jogo 4 (JG4)"]
df_resultado_jogo_5 = df_resultado[df_resultado["Jogo"] == "Jogo 5 (JG5)"]
df_resultado_jogo_6 = df_resultado[df_resultado["Jogo"] == "Jogo 6 (JG6)"]
df_resultado_jogo_7 = df_resultado[df_resultado["Jogo"] == "Jogo 7 (JG7)"]
df_resultado_jogo_8 = df_resultado[df_resultado["Jogo"] == "Jogo 8 (JG8)"]

display(df_resultado_jogo_1, df_resultado_jogo_2, df_resultado_jogo_3, df_resultado_jogo_4,
         df_resultado_jogo_5, df_resultado_jogo_6, df_resultado_jogo_7, df_resultado_jogo_8)


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
0,Jogo 1 (JG1),JUV. KP,3,1,0,0,117.04,97.64,19.40,1
1,Jogo 1 (JG1),KillerColorado,0,0,0,1,97.64,117.04,-19.40,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
2,Jogo 2 (JG2),Máquina Laranjja,3,1,0,0,123.04,104.29,18.75,1
3,Jogo 2 (JG2),DM Studio,0,0,0,1,104.29,123.04,-18.75,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
4,Jogo 3 (JG3),dasdoresfc,3,1,0,0,119.54,110.35,9.19,1
5,Jogo 3 (JG3),Tatols Beants F.C,0,0,0,1,110.35,119.54,-9.19,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
7,Jogo 4 (JG4),JV5 Tricolor Gaúcho,3,1,0,0,115.44,103.07,12.37,1
6,Jogo 4 (JG4),Grêmio imortal 36,0,0,0,1,103.07,115.44,-12.37,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
8,Jogo 5 (JG5),FÚRIA LEON,3,1,0,0,116.79,106.39,10.40,1
9,Jogo 5 (JG5),Texas Club 2026,0,0,0,1,106.39,116.79,-10.40,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
11,Jogo 6 (JG6),TORRESMO COM PINGA PRO26.1,3,1,0,0,104.29,99.64,4.65,1
10,Jogo 6 (JG6),AZURRA82,0,0,0,1,99.64,104.29,-4.65,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
13,Jogo 7 (JG7),A Lenda Super Vasco F.c,3,1,0,0,120.79,103.59,17.20,1
12,Jogo 7 (JG7),Grêmio imortal 37,0,0,0,1,103.59,120.79,-17.20,2


,Jogo,Nome do Time,Pontos,Vitórias,Empates,Derrotas,Total Cartola,Cartola Sofrido,Saldo Cartola,Posição
15,Jogo 8 (JG8),Mau Humor F.C.,3,1,0,0,122.49,104.89,17.60,1
14,Jogo 8 (JG8),Super Vasco f.c,0,0,0,1,104.89,122.49,-17.60,2


In [16]:
# Verifica se a variável df_resultado_por_grupo existe e está populada
if 'df_resultado_por_jogo' in locals() and df_resultado_por_jogo:
    # Criar estrutura em formato de dicionário para JSON/JS
    classificacao_js = {}

    for jogo, df in df_resultado_por_jogo.items():
        classificacao_js[jogo] = []
        for _, row in df.iterrows():
            classificacao_js[jogo].append({
                "posicao": int(row["Posição"]),
                "nome": row["Nome do Time"],
                "pontos": int(row["Pontos"]),
                "vitorias": int(row["Vitórias"]),
                "empates": int(row["Empates"]),
                "derrotas": int(row["Derrotas"]),
                "totalCartola": float(row["Total Cartola"]),
                "cartolaSofrido": float(row["Cartola Sofrido"]),
                "saldoCartola": float(row["Saldo Cartola"])
            })

    # Converter para JSON formatado
    json_str = json.dumps(classificacao_js, indent=2, ensure_ascii=False)

    # Salvar como arquivo JS com uma variável global
    with open("1_classificacao_por_jogo_oitavas_sula.js", "w", encoding="utf-8") as f:
        f.write("const classificacao_oitavas_sula = ")
        f.write(json_str)
        f.write(";")

    print("Arquivo JS salvo com sucesso.")

else:
    print("Classificação das Oitavas da Sula indisponível. O mercado pode estar em manutenção ou os dados ainda não foram gerados.")


Arquivo JS salvo com sucesso.


In [17]:
def exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada, jogo=None):
    """
    Exibe os resultados de uma rodada específica, com pontuação e dados dos times.
    """

    if "Rodada" not in df_rodadas.columns:
        raise ValueError("O DataFrame df_rodadas precisa conter a coluna 'Rodada'.")

    if rodada not in df_rodadas["Rodada"].values:
        return pd.DataFrame([{
            "Jogo": jogo or "-",
            "Rodada": rodada,
            "Mandante_Nome": "-",
            "Mandante_Clube": "-",
            "Mandante_Participante": "-",
            "Mandante_Pontos": "-",
            "Visitante_Nome": "-",
            "Visitante_Clube": "-",
            "Visitante_Participante": "-",
            "Visitante_Pontos": "-",
        }])

    df_filtrado = df_rodadas[df_rodadas["Rodada"] == rodada]
    if jogo:
        df_filtrado = df_filtrado[df_filtrado["Jogo"] == jogo]

    resultados = []

    for _, row in df_filtrado.iterrows():
        jogo_ = row["Jogo"]
        mandante = row["Mandante_Nome"]
        visitante = row["Visitante_Nome"]

        pontos_mandante = df_pontuacoes.get(f"Rodada {rodada}", {}).get(mandante, None)
        pontos_visitante = df_pontuacoes.get(f"Rodada {rodada}", {}).get(visitante, None)

        resultados.append({
            "Jogo": jogo_,
            "Rodada": rodada,
            "Mandante_Nome": mandante,
            "Mandante_Pontos": pontos_mandante,
            "Visitante_Nome": visitante,
            "Visitante_Pontos": pontos_visitante
        })

    return pd.DataFrame(resultados)

In [18]:
# Exibir resultados da 13ª rodada
df_resultados_rodada_13 = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=13)

# Exibir apenas os resultados do Grupo B na 1ª rodada
df_resultados_jogo_2 = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=13, jogo="Jogo 2 (JG2)")

# Exibir
display(df_resultados_rodada_13)

,Jogo,Rodada,Mandante_Nome,Mandante_Pontos,Visitante_Nome,Visitante_Pontos
0,Jogo 1 (JG1),13,JUV. KP,117.04,KillerColorado,97.64
1,Jogo 2 (JG2),13,Máquina Laranjja,123.04,DM Studio,104.29
2,Jogo 3 (JG3),13,dasdoresfc,119.54,Tatols Beants F.C,110.35
3,Jogo 4 (JG4),13,Grêmio imortal 36,103.07,JV5 Tricolor Gaúcho,115.44
4,Jogo 5 (JG5),13,FÚRIA LEON,116.79,Texas Club 2026,106.39
5,Jogo 6 (JG6),13,AZURRA82,99.64,TORRESMO COM PINGA PRO26.1,104.29
6,Jogo 7 (JG7),13,Grêmio imortal 37,103.59,A Lenda Super Vasco F.c,120.79
7,Jogo 8 (JG8),13,Super Vasco f.c,104.89,Mau Humor F.C.,122.49


In [19]:
# Criar arquivo com uma aba para cada rodada contendo os resultados detalhados
from pathlib import Path

# Caminho do arquivo de saída
caminho_resultados = "1_resultados_oitavas_sula.xlsx"

# Descobrir as rodadas únicas no DataFrame
rodadas_disponiveis = sorted(df_rodadas["Rodada"].unique())

with pd.ExcelWriter(caminho_resultados) as writer:
    for rodada in rodadas_disponiveis:
        df_resultados = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=rodada)
        nome_aba = f"Rodada {rodada}"
        df_resultados.to_excel(writer, sheet_name=nome_aba, index=False)

print(f"Arquivo salvo com sucesso: {Path(caminho_resultados).resolve()}")

display(df_resultados)

Arquivo salvo com sucesso: C:\Users\ferna\Projetos\GitHub\cartola_2026\sulamericana\datasets_sula\1_resultados_oitavas_sula.xlsx


,Jogo,Rodada,Mandante_Nome,Mandante_Pontos,Visitante_Nome,Visitante_Pontos
0,Jogo 1 (JG1),14,KillerColorado,NaN,JUV. KP,NaN
1,Jogo 2 (JG2),14,DM Studio,NaN,Máquina Laranjja,NaN
2,Jogo 3 (JG3),14,Tatols Beants F.C,NaN,dasdoresfc,NaN
3,Jogo 4 (JG4),14,JV5 Tricolor Gaúcho,NaN,Grêmio imortal 36,NaN
4,Jogo 5 (JG5),14,Texas Club 2026,NaN,FÚRIA LEON,NaN
5,Jogo 6 (JG6),14,TORRESMO COM PINGA PRO26.1,NaN,AZURRA82,NaN
6,Jogo 7 (JG7),14,A Lenda Super Vasco F.c,NaN,Grêmio imortal 37,NaN
7,Jogo 8 (JG8),14,Mau Humor F.C.,NaN,Super Vasco f.c,NaN


In [20]:
resultados_js = []

for rodada in sorted(df_rodadas["Rodada"].unique()):
    df_resultados = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=rodada)
    
    for _, row in df_resultados.iterrows():
        resultado = {
            "jogo": row["Jogo"],
            "rodada": int(rodada),
            "mandante": {
                "nome": row["Mandante_Nome"],
                "pontos": float(row["Mandante_Pontos"]) if row["Mandante_Pontos"] is not None else None
            },
            "visitante": {
                "nome": row["Visitante_Nome"],
                "pontos": float(row["Visitante_Pontos"]) if row["Visitante_Pontos"] is not None else None
            },
            "vencedor": (
                "mandante" if row["Mandante_Pontos"] is not None and row["Visitante_Pontos"] is not None and row["Mandante_Pontos"] > row["Visitante_Pontos"]
                else "visitante" if row["Mandante_Pontos"] is not None and row["Visitante_Pontos"] is not None and row["Mandante_Pontos"] < row["Visitante_Pontos"]
                else "empate" if row["Mandante_Pontos"] == row["Visitante_Pontos"] and row["Mandante_Pontos"] is not None
                else "indefinido"
            )

        }
        resultados_js.append(resultado)

# Exportar para arquivo .js
import json

with open("1_resultados_oitavas_sula.js", "w", encoding="utf-8") as f:
    f.write("const resultados_oitavas_sula = ")
    f.write(json.dumps(resultados_js, indent=2, ensure_ascii=False))
    f.write(";")

try:
    rodada_ref = int(rod_ref)
except Exception:
    rodada_ref = 0

parcial_payload = {"rodada": rodada_ref, "times": {}}
try:
    col_parcial = f"Rodada {rodada_ref}"
    if col_parcial in df_pontuacoes.columns:
        times_map = {}
        for nome in df_pontuacoes.index:
            if nome not in ids_times:
                continue
            try:
                val = df_pontuacoes.at[nome, col_parcial]
            except Exception:
                continue
            if str(val) in ("", "nan"):
                continue
            try:
                times_map[str(ids_times[nome])] = float(val)
            except Exception:
                continue
        parcial_payload["times"] = times_map
except Exception:
    pass

try:
    status_http_val = int(status_http)
except Exception:
    status_http_val = None

sula_meta = {
    "rodada_atual": rodada_ref,
    "parcial_disponivel": bool(status_http_val == 2 and parcial_payload["times"])
}

with open("1_resultados_oitavas_sula.js", "a", encoding="utf-8") as f:
    f.write("const pontuacaoParcialRodadaAtual = ")
    f.write(json.dumps(parcial_payload, indent=2, ensure_ascii=False))
    f.write(";")
    f.write("window.sulaMeta = ")
    f.write(json.dumps(sula_meta, indent=2, ensure_ascii=False))
    f.write(";")


### Identificando Vencedores das Oitavas de Final

In [21]:
def obter_classificados_com_id(resultados, ids_por_nome):
    classificados = []
    jogos_agrupados = {}
    for jogo in resultados:
        chave = jogo['jogo']
        if chave not in jogos_agrupados:
            jogos_agrupados[chave] = []
        jogos_agrupados[chave].append(jogo)

    for chave, partidas in jogos_agrupados.items():
        if len(partidas) < 2:
            continue

        partidas = sorted(partidas, key=lambda x: x['rodada'])
        ida, volta = partidas

        if ida['mandante']['pontos'] is None or volta['mandante']['pontos'] is None:
            continue

        time1 = ida['mandante']['nome']
        time2 = ida['visitante']['nome']

        pontos_time1 = ida['mandante']['pontos'] + volta['visitante']['pontos']
        pontos_time2 = ida['visitante']['pontos'] + volta['mandante']['pontos']

        if pontos_time1 > pontos_time2:
            vencedor = time1
        elif pontos_time2 > pontos_time1:
            vencedor = time2
        else:
            vencedor = "EMPATE"

        classificados.append({
            "jogo": chave,
            "classificado_nome": vencedor,
            "classificado_id": ids_por_nome.get(vencedor) if vencedor in ids_por_nome else None
        })

    return classificados


In [22]:
if not COMPETICAO_INICIOU:
    print("Competi??o ainda n?o come?ou. Pulando gera??o de classificados.")
else:
    def carregar_payload_js(caminho_arquivo, nome_constante):
        conteudo = Path(caminho_arquivo).read_text(encoding="utf-8")
        prefixo = f"const {nome_constante} = "
        if prefixo not in conteudo:
            raise ValueError(f"Constante '{nome_constante}' n?o encontrada em {caminho_arquivo}")

        payload = conteudo.split(prefixo, 1)[1].lstrip()
        delimitador_final = "]" if payload.startswith("[") else "}"
        profundidade = 0
        fim = None

        for i, caractere in enumerate(payload):
            if caractere in "[{":
                profundidade += 1
            elif caractere in "]}":
                profundidade -= 1
                if profundidade == 0 and caractere == delimitador_final:
                    fim = i + 1
                    break

        if fim is None:
            raise ValueError(f"N?o foi poss?vel extrair o JSON de {caminho_arquivo}")

        return json.loads(payload[:fim])

    ids_por_nome = {v: k for k, v in nomes_por_id.items()}
    resultados = carregar_payload_js("1_resultados_oitavas_sula.js", "resultados_oitavas_sula")

    classificados = obter_classificados_com_id(resultados, ids_por_nome)

    df_classificados = pd.DataFrame(classificados)
    df_classificados.to_excel("1_classificados_oitavas_sula.xlsx", index=False)

    with open("1_classificados_oitavas_sula.js", "w", encoding="utf-8") as f:
        f.write("const classificados_oitavas_sula = ")
        json.dump(classificados, f, ensure_ascii=False, indent=2)
        f.write(";")
    print("Classificados salvos com sucesso em '1_classificados_oitavas_sula.xlsx' e '1_classificados_oitavas_sula.js'.")
    print("Classificados encontrados:", classificados)


Classificados salvos com sucesso em '1_classificados_oitavas_sula.xlsx' e '1_classificados_oitavas_sula.js'.
Classificados encontrados: [{'jogo': 'Jogo 1 (JG1)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 2 (JG2)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 3 (JG3)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 4 (JG4)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 5 (JG5)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 6 (JG6)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 7 (JG7)', 'classificado_nome': 'EMPATE', 'classificado_id': None}, {'jogo': 'Jogo 8 (JG8)', 'classificado_nome': 'EMPATE', 'classificado_id': None}]
